# 09 — Image Similarity Search with LanceDB & CLIP

In this notebook, we will build a simple system that can search for images using **Natural Language** (e.g. "A fluffy orange cat").

We'll use **CLIP** (Contrastive Language-Image Pre-training), a model that understands both text and images in the same "meaning space" (vector space).

## 1) Setup & Loading Data

First, we'll import our tools and load any animal images we've placed in the `images/` folder.

In [ ]:
import os
from pathlib import Path
import lancedb
from PIL import Image
from sentence_transformers import SentenceTransformer
import matplotlib.pyplot as plt

# 1. Setup paths
BASE_DIR = Path.cwd()
IMAGE_DIR = BASE_DIR / "images"
DB_PATH = str(BASE_DIR / "lancedb_images")

# 2. Find and preview images
image_paths = list(IMAGE_DIR.glob("*.png")) + list(IMAGE_DIR.glob("*.jpg"))
print(f"Found {len(image_paths)} images.")

fig, axes = plt.subplots(1, len(image_paths), figsize=(15, 5))
for i, p in enumerate(image_paths):
    axes[i].imshow(Image.open(p))
    axes[i].axis('off')
plt.show()

## 🛠️ Step 1: Load the CLIP Model

CLIP is a multi-modal model. It can turn an image into a 512-dimension vector, and it can turn a piece of text into a vector in that *same* 512-dimension space.

In [ ]:
# Load the CLIP ViT-B-32 model
model = SentenceTransformer('clip-ViT-B-32')
print("Model loaded successfully!")

## 🛠️ Step 2: Generate Vectors (Embeddings)

Next, we'll turn our images into vectors. 

> **Note**: The model doesn't see the filename or any text labels. It only processes the raw pixels to create a mathematical representation of the image's content.

In [ ]:
print("Generating embeddings...")
images = [Image.open(p) for p in image_paths]

vectors = model.encode(images)

print(f"Generated {len(vectors)} vectors with {len(vectors[0])} dimensions each.")

## 3) Storage (LanceDB)

We'll now pair these vectors with their filenames and store them in **LanceDB**.

In [ ]:
# Pair images and vectors
data = [
    {"path": str(p), "vector": v.tolist()}
    for p, v in zip(image_paths, vectors)
]

# Connect to LanceDB and create a search table
if os.path.exists(DB_PATH):
    import shutil
    shutil.rmtree(DB_PATH)

db = lancedb.connect(DB_PATH)
table = db.create_table("images", data=data)

print("Knowledge base ready for search!")

## 🛠️ Step 3: Semantic Search

Finally, we can search for images using a simple text query.

In [ ]:
def search_by_text(query, limit=1):
    # Embed the text query
    query_vector = model.encode([query])[0]
    # Ask the database for the closest match
    results = table.search(query_vector).limit(limit).to_pandas()
    
    # Display the result
    for _, row in results.iterrows():
        img = Image.open(row['path'])
        plt.imshow(img)
        plt.title(f"Query: '{query}'\nMatch: {Path(row['path']).name} (Distance: {row['_distance']:.4f})")
        plt.axis('off')
        plt.show()

search_by_text("a photo of a cat")

### 🎉 Summary
1. We used **CLIP** to map images/text to vectors.
2. We used **LanceDB** to store and search those vectors.
3. We demonstrated that vector search can find photos based on **content**!